In [14]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()
using Revise, Distributions, Dates, DataFrames, CSV, DataFramesMeta, LinearAlgebra, FFTW
using TrypColonies
import Serialization as s

  Activating project at `c:\Users\Andreas\Dropbox\phd\TrypColonies\TrypColonies`


In [15]:
data_path = find_data_path(local_data = false)

"C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\"

In [10]:
data_base = s.deserialize("data_base\\df.jls") 
data_base = @subset(data_base, :location .== data_path)   
first(data_base ,5)

Row,radius_tanget,scale_fac,agent_number,noise_dis,velo_var_relative,sweep_parameter,sampling_interval,L,path,walker_step_size,unix_date,run_time,location,name,N,parameter_increment,Diff_coe,adsorption_rate,storage,noise_strength,grid_recover_rate,iterations,growth_rate,Diameter_colony,parameter_steps,velocity_dis,radius_collision,date,walker_speed_phy,decay_rate,grid_strength,total_time,doubling_time,Δt_walker_min,sweep_para_start_value
,Int64,Int64,Int64,UnionAll,Float64,String,Int64,Tuple…,String?,Int64,Float64?,Float64?,String?,String?,Tuple…,Float64,Float64,Float64,Float64?,Float64,Float64,Int64,Float64?,Float64,Int64,UnionAll,Int64,DateTime?,Float64,Float64,Int64,Float64,Float64?,Float64?,Real?
1,1,4,250000,Normal,1.5,radius_tanget,1200,"(0.015, 0.015)",C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\radius_tanget_sweep_threading_16_32_250000_time_seed_3543,3,1.73843e9,76925.0,C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\,radius_tanget_sweep_threading_16_32,"(1000, 1000)",2.0,7.0e-11,0.001,65.2002,1.2,15.0,1,1.157e-5,0.003,16,Normal,12,2025-02-01T17:34:38.663,5.0e-6,0.001,350,36000.0,16.6415,missing,1
2,15,4,250000,Normal,1.5,adsorption_rate,1200,"(0.015, 0.015)",C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\adsorption_right_noise_very_hard_walls_sweep_threading_16_32_250000_time_seed_3435,3,1.73996e9,68009.3,C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\,adsorption_right_noise_very_hard_walls_sweep_threading_16_32,"(1000, 1000)",5.0e-5,7.0e-11,1.0e-5,65.1349,1.2,15.0,1,1.157e-5,0.003,16,Normal,14,2025-02-19T10:37:26.282,5.0e-6,0.001,8000,36000.0,16.6415,missing,1.0e-5
3,15,4,250000,Normal,1.5,radius_collision,1200,"(0.015, 0.015)",C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\radisu_collision_random_movement_sweep_threading_16_32_250000_time_seed_3854,3,1.74014e9,65967.0,C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\,radisu_collision_random_movement_sweep_threading_16_32,"(1000, 1000)",1.0,7.0e-11,1.0e-5,65.1882,2.2,15.0,1,1.157e-5,0.003,16,Normal,6,2025-02-21T12:10:23.668,5.0e-6,0.001,8000,36000.0,16.6415,missing,6
4,15,4,250000,Normal,1.5,radius_collision,1200,"(0.015, 0.015)",C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\radisu_collision_random_movement_modified_collision_sweep_threading_16_32_250000_time_seed_7315,3,1.74022e9,65754.7,C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\,radisu_collision_random_movement_modified_collision_sweep_threading_16_32,"(1000, 1000)",1.0,7.0e-11,1.0e-5,65.1855,2.2,15.0,1,1.157e-5,0.003,16,Normal,6,2025-02-22T11:17:52.284,5.0e-6,0.001,8000,36000.0,16.6415,missing,6
5,15,4,250000,Normal,1.5,radius_collision,1200,"(0.015, 0.015)",C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\radisu_collision_random_movement_diamond_collision_sweep_threading_16_32_250000_time_seed_8444,3,1.74117e9,65217.5,C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\,radisu_collision_random_movement_diamond_collision_sweep_threading_16_32,"(1000, 1000)",1.0,7.0e-11,1.0e-5,65.1859,2.2,15.0,1,1.157e-5,0.003,16,Normal,6,2025-03-05T11:21:04.241,5.0e-6,0.001,8000,36000.0,16.6415,missing,6


## Calculate all metrics

This section computes and stores metric time series for each simulation in the selected `data_base` rows.

### What is computed per run

- `adsorb`: total adsorbed material
- `cell_gain`: relative increase in cell count (scaled by initial agent number)
- `area_gain`: relative area increase
- `angular_metric`, `pair_metric`, `radial_density`: intermediate spatial descriptors
- `cv`, `W`, `fk`, `max_p1`: derived morphology/roughness metrics
- `time`: time axis in hours from `sampling_interval`

### Input requirements

- `data_base` must contain valid local paths.
- Raw simulation files must be present (not metadata-only entries).
- The metric output folder is created automatically at `analysis/metrics`.

### Output

Each parameter/iteration combination is saved as one serialized metric file (`.jls`) in the metrics folder.
A length-consistency check is applied before saving; inconsistent results are skipped with an error message.

### Note

This computation will take roughly 10 min for one parameter sweep with default parameters on a Julia kernek with 16 threads and a cpu with 16 physical cores. Expect much more on more on a less potentent hardware. 

In [12]:
first(data_base, 4)

Row,radius_tanget,scale_fac,agent_number,noise_dis,velo_var_relative,sweep_parameter,sampling_interval,L,path,walker_step_size,unix_date,run_time,location,name,N,parameter_increment,Diff_coe,adsorption_rate,storage,noise_strength,grid_recover_rate,iterations,growth_rate,Diameter_colony,parameter_steps,velocity_dis,radius_collision,date,walker_speed_phy,decay_rate,grid_strength,total_time,doubling_time,Δt_walker_min,sweep_para_start_value
,Int64,Int64,Int64,UnionAll,Float64,String,Int64,Tuple…,String?,Int64,Float64?,Float64?,String?,String?,Tuple…,Float64,Float64,Float64,Float64?,Float64,Float64,Int64,Float64?,Float64,Int64,UnionAll,Int64,DateTime?,Float64,Float64,Int64,Float64,Float64?,Float64?,Real?
1,1,4,250000,Normal,1.5,radius_tanget,1200,"(0.015, 0.015)",C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\radius_tanget_sweep_threading_16_32_250000_time_seed_3543,3,1.73843e9,76925.0,C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\,radius_tanget_sweep_threading_16_32,"(1000, 1000)",2.0,7.0e-11,0.001,65.2002,1.2,15.0,1,1.157e-5,0.003,16,Normal,12,2025-02-01T17:34:38.663,5.0e-6,0.001,350,36000.0,16.6415,missing,1
2,15,4,250000,Normal,1.5,adsorption_rate,1200,"(0.015, 0.015)",C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\adsorption_right_noise_very_hard_walls_sweep_threading_16_32_250000_time_seed_3435,3,1.73996e9,68009.3,C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\,adsorption_right_noise_very_hard_walls_sweep_threading_16_32,"(1000, 1000)",5.0e-5,7.0e-11,1.0e-5,65.1349,1.2,15.0,1,1.157e-5,0.003,16,Normal,14,2025-02-19T10:37:26.282,5.0e-6,0.001,8000,36000.0,16.6415,missing,1.0e-5
3,15,4,250000,Normal,1.5,radius_collision,1200,"(0.015, 0.015)",C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\radisu_collision_random_movement_sweep_threading_16_32_250000_time_seed_3854,3,1.74014e9,65967.0,C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\,radisu_collision_random_movement_sweep_threading_16_32,"(1000, 1000)",1.0,7.0e-11,1.0e-5,65.1882,2.2,15.0,1,1.157e-5,0.003,16,Normal,6,2025-02-21T12:10:23.668,5.0e-6,0.001,8000,36000.0,16.6415,missing,6
4,15,4,250000,Normal,1.5,radius_collision,1200,"(0.015, 0.015)",C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\radisu_collision_random_movement_modified_collision_sweep_threading_16_32_250000_time_seed_7315,3,1.74022e9,65754.7,C:\\Users\\Andreas\\Desktop\\simulation_data_julia_knecht\\,radisu_collision_random_movement_modified_collision_sweep_threading_16_32,"(1000, 1000)",1.0,7.0e-11,1.0e-5,65.1855,2.2,15.0,1,1.157e-5,0.003,16,Normal,6,2025-02-22T11:17:52.284,5.0e-6,0.001,8000,36000.0,16.6415,missing,6


In [ ]:
for jj in 1:nrow(data_base)
    data_used = DataFrame(data_base[jj,:]) 
    #data_used = DataFrame(data_base[5,:]) 
    path_vec = create_data_path_vector(data_used.path)

    """
    if isdir(data_used.path[1]*"\\analysis\\metrics")
        println("metrics already calculated for this parameter set, skipping")
    else
        """
        
        println("calculating metrics for this parameter set" , data_used.path[1][end-30:end])
        data_s = s.deserialize(path_vec[1][1])
        datu_s = transform_data(data_s)
        Og_size = og_size(datu_s[1]...)
        length_data = length(datu_s)
        para_range = collect(1:data_used.parameter_steps[1]*data_used.iterations[1])

        mkpath(data_used.path[1]*"\\analysis\\metrics")

        Threads.@threads for i in 1:length(para_range)
            data = s.deserialize(path_vec[1][i])
            datu = transform_data(data)
            metric_dict = Dict{Symbol,Any}()
            metric_dict[:adsorb] = [adsorbed_material(x...) for x in datu]
            metric_dict[:cell_gain] = ([cell_gain(x...) for x in datu].+1)*data_used.agent_number[1]
            metric_dict[:area_gain] = [area_gain(x...) for x in datu]

            metric_dict[:angular_metric] = [angular_metric(x...) for x in datu]
            metric_dict[:pair_metric] = [pair_cor_metric(x... ; max_samples = 10000000) for x in datu]
            metric_dict[:radial_density] = [radial_density_c(x...) for x in datu]
            
            metric_dict[:time] = [data_used.sampling_interval[1]/3600 *j for j in 0:(length_data-1)]
            metric_dict[:cv] = Float64.([(std(x)/mean(x))>0 ? (mean(x) != 0 ? std(x)/mean(x) : 0) : 0 for x in metric_dict[:angular_metric] ])
            metric_dict[:W] = [roughness(x,Og_size) for x in metric_dict[:angular_metric]]
            metric_dict[:fk] = [fourier_ik(x,Og_size) for x in metric_dict[:angular_metric]]
            metric_dict[:max_p1] = Float64.([(maximum(x)-mean(x))/mean(x) > 0 ? (mean(x) != 0 ? (maximum(x)-mean(x))/mean(x) : 0) : 0 for x in metric_dict[:pair_metric] ])

            if length(unique([length(metric_dict[key]) for key in keys(metric_dict)])) == 1
                offset =100000
                s.serialize(data_used.path[1]*"\\analysis\\metrics\\"*string(i+offset)*".jls", metric_dict)
            else
                println("Error: metric lengths not equal for parameter set: ", i, " data is not saved")
            end
            
            
        end
    end
    



#end


calculating metrics for this parameter set_16_16_32_250000_time_seed_4177


LoadError: TaskFailedException

[91m    nested task error: [39mDimensionMismatch: a has size (4000, 4000), b has size (1, 0), mismatch at dim 1
    Stacktrace:
      [1] [0m[1mthrow_promote_shape_mismatch[22m[0m[1m([22m[90ma[39m::[0mTuple[90m{Base.OneTo{…}, Base.OneTo{…}}[39m, [90mb[39m::[0mTuple[90m{Base.OneTo{…}, Base.OneTo{…}}[39m, [90mi[39m::[0mInt64[0m[1m)[22m
    [90m    @[39m [90mBase[39m [90m.\[39m[90m[4mindices.jl:135[24m[39m
      [2] [0m[1mpromote_shape[22m
    [90m    @[39m [90m.\[39m[90m[4mindices.jl:196[24m[39m[90m [inlined][39m
      [3] [0m[1mpromote_shape[22m
    [90m    @[39m [90m.\[39m[90m[4mindices.jl:188[24m[39m[90m [inlined][39m
      [4] [0m[1m-[22m[0m[1m([22m[90mA[39m::[0mMatrix[90m{Int64}[39m, [90mB[39m::[0mMatrix[90m{Int64}[39m[0m[1m)[22m
    [90m    @[39m [90mBase[39m [90m.\[39m[90m[4marraymath.jl:7[24m[39m
      [5] [0m[1mangular_metric[22m[0m[1m([22m[90magent_lis

## Add a new metric to existing metric files

Use this section when metric files already exist and you want to append one additional metric without recomputing everything.

### Typical use case

- Define the metric key in `new_m` (for example `:time`).
- Loop through selected `data_base` entries and all parameter/iteration files.
- Load each existing metric dictionary, add the new metric, and save it back in place.

### Safety notes

- Keep the new metric vector length consistent with the existing time series length.
- Test on a small subset first (one row or one parameter index) before running a larger batch.
- Consider creating a backup if you are overwriting many metric files.

In [26]:
new_m = :time

#for j in (nrow(data_base)-1):nrow(data_base)
for j in 40:40
    data_used = DataFrame(data_base[j,:]) 
    path_vec = create_data_path_vector(data_used.path)
    metric_path_vec = create_metric_path_vector(data_used.path)
    metric_dict = s.deserialize(metric_path_vec[1][1])
    """
    if haskey(metric_dict, new_m)
        println("metric already calculated for this parameter set, skipping")
    else
        println("calculating new metric $(new_m) for this parameter set")
        """
        data = s.deserialize(path_vec[1][1])
        datu = transform_data(data)
        Og_size = og_size(datu[1]...)
        length_data = round(Int,data_used.total_time[1]/data_used.sampling_interval[1])
        para_range = collect(1:data_used.parameter_steps[1]*data_used.iterations[1])


        Threads.@threads for i in 1:length(para_range)
            data = s.deserialize(path_vec[1][i])
            datu = transform_data(data)
            metric_dict = s.deserialize(metric_path_vec[1][i])

            metric_dict[new_m] = [data_used.sampling_interval[1]/3600 *x for x in 0:(length_data-1)]
            s.serialize(metric_path_vec[1][i], metric_dict)
            
            
        
    end
    



end


46